In [ ]:
import os
import pyodbc
import dotenv
import pandas as pd
import mlflow
import mlflow.sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

dotenv.load_dotenv()

SEED = 42
np.random.seed(SEED)

# Connection string using the specific InterSystems ODBC driver
# Note: Ensure the driver name exactly matches what is installed on your client
# Updated with the correct driver name from your system
connection_string = (
    f"DRIVER={{InterSystems IRIS ODBC35}};"
    f"SERVER={os.getenv('IRIS_SERVER')};"
    f"PORT={os.getenv('IRIS_PORT')};"
    f"DATABASE={os.getenv('IRIS_NAMESPACE')};"
    f"UID={os.getenv('IRIS_USERNAME')};"
    f"PWD={os.getenv('IRIS_PASSWORD')}"
)

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("LinearRegression-experimentation")
mlflow.sklearn.autolog(log_models=True, exclusive=False)

with pyodbc.connect(connection_string) as cnxn:
    df =  pd.read_sql(f"SELECT * FROM MLpipeline.PointSamples", cnxn)

df_train, df_test = train_test_split(df, test_size=0.2, random_state=SEED)

X_train = df_train[["x"]]
y_train = df_train["y"]
X_test = df_test[["x"]]
y_test = df_test["y"]

params = {"fit_intercept": True}

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression(**params))
])


with mlflow.start_run():
    mlflow.log_param("random_state", SEED)
    pipeline.fit(X_train, y_train)


    # Calculate metrics
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    
    # Log manually with custom keys
    mlflow.log_metrics({
        "train_mae": mean_absolute_error(y_train, y_pred_train),
        "val_mae": mean_absolute_error(y_test, y_pred_test),
        "train_r2": r2_score(y_train, y_pred_train),
        "val_r2": r2_score(y_test, y_pred_test)
    })


In [ ]:

line_x = np.linspace(df["x"].min(), df["x"].max(), 100).reshape(-1, 1)
line_y = pipeline.predict(line_x)

plt.figure(figsize=(10, 6))
plt.scatter(df["x"], df["y"], label='Data points', alpha=0.6)
plt.plot(line_x, line_y, color='red', label='Fitted line', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()